# Giáo trình Dữ liệu lớn – Chương 7

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume và thay `data/` bằng `/Volumes/<catalog>/<schema>/<volume>/`.

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch{ch:02d}/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
!pip install -q pyspark==3.5.7 pyarrow==16.1.0 pandas==2.2.2
import os
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch07").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 7.1. Khảo sát WSSSE và hệ số silhouette theo số cụm K.


> Đoạn mã 7.1 cần `df_features` (vector đặc trưng đã chuẩn hóa). Ô chuẩn bị bên dưới dựng `df_features` từ `data/khach_hang.csv` theo đúng cách của Đoạn mã 7.2.


In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
df_kh = spark.read.csv("data/khach_hang.csv", header=True, inferSchema=True)
asm = VectorAssembler(inputCols=["thu_nhap", "diem_chi_tieu", "so_don_hang"], outputCol="features_raw")
sc_model = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True).fit(asm.transform(df_kh))
df_features = sc_model.transform(asm.transform(df_kh)).select("ma_kh", "features").cache()
print(df_features.count(), "khach hang")

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(metricName="silhouette",
                                distanceMeasure="squaredEuclidean")

for k in range(2, 11):
    kmeans = KMeans(featuresCol="features", k=k, seed=42)
    model = kmeans.fit(df_features)
    du_doan = model.transform(df_features)
    wssse = model.summary.trainingCost  # ham muc tieu WSSSE
    sil = evaluator.evaluate(du_doan)   # silhouette trung binh
    print("k = %d | WSSSE = %.1f | silhouette = %.3f"
          % (k, wssse, sil))

## Đoạn mã 7.2. Quy trình phân cụm khách hàng hoàn chỉnh với Pipeline.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = (SparkSession.builder.appName("PhanCumKhachHang")
         .getOrCreate())

# Cot du lieu: ma_kh, thu_nhap, diem_chi_tieu, so_don_hang
df = spark.read.csv("data/khach_hang.csv",
                    header=True, inferSchema=True)

assembler = VectorAssembler(
    inputCols=["thu_nhap", "diem_chi_tieu", "so_don_hang"],
    outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw",
                        outputCol="features",
                        withMean=True, withStd=True)
kmeans = KMeans(featuresCol="features", predictionCol="prediction",
                k=3, initMode="k-means||", maxIter=50, seed=42)

pipeline = Pipeline(stages=[assembler, scaler, kmeans])
model = pipeline.fit(df)
du_doan = model.transform(df)

evaluator = ClusteringEvaluator(metricName="silhouette")
print("Silhouette =", evaluator.evaluate(du_doan))

# Dem so khach hang cua tung cum
du_doan.groupBy("prediction").count().orderBy("prediction").show()

## Đoạn mã 7.3. Khai phá tập mục phổ biến và luật kết hợp bằng FPGrowth.


In [ ]:
from pyspark.ml.fpm import FPGrowth

giao_dich = spark.createDataFrame([
    (1, ["banh mi", "sua", "trung"]),
    (2, ["banh mi", "sua"]),
    (3, ["trung", "ca phe"]),
    (4, ["banh mi", "sua", "ca phe"]),
    (5, ["ca phe", "trung"]),
    (6, ["banh mi", "trung"]),
], ["ma_gd", "items"])

fp = FPGrowth(itemsCol="items", minSupport=0.3, minConfidence=0.6)
fp_model = fp.fit(giao_dich)

# 1) Cac tap muc pho bien va tan suat xuat hien
tap_pho_bien = fp_model.freqItemsets
tap_pho_bien.orderBy("freq", ascending=False).show(truncate=False)

# 2) Cac luat ket hop kem confidence, lift, support
luat = fp_model.associationRules
luat.orderBy("lift", ascending=False).show(truncate=False)

# 3) Ap luat len tung gio hang de goi y mat hang mua kem
fp_model.transform(giao_dich).show(truncate=False)

## Đoạn mã 7.4. Huấn luyện ALS trên MovieLens và sinh gợi ý.


> Cần bộ dữ liệu MovieLens *ml-latest-small* (ô chuẩn bị tải về `data/movielens/ratings.csv`; giấy phép GroupLens không cho phép phân phối lại nên kho mã không kèm tệp này).


In [ ]:
import os, urllib.request, zipfile
if not os.path.exists("data/movielens/ratings.csv"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml-latest-small.zip")
    with zipfile.ZipFile("ml-latest-small.zip") as z:
        z.extract("ml-latest-small/ratings.csv", ".")
    os.makedirs("data/movielens", exist_ok=True)
    os.replace("ml-latest-small/ratings.csv", "data/movielens/ratings.csv")
print("ratings.csv san sang")

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

ratings = spark.read.csv("data/movielens/ratings.csv",
                         header=True, inferSchema=True) \
                .select("userId", "movieId", "rating")

train, test = ratings.randomSplit([0.8, 0.2], seed=42)

als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
          rank=10, regParam=0.1, maxIter=15,
          implicitPrefs=False, coldStartStrategy="drop", seed=42)
als_model = als.fit(train)

# Danh gia RMSE tren tap kiem tra
du_doan = als_model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse",
                                labelCol="rating",
                                predictionCol="prediction")
print("RMSE =", evaluator.evaluate(du_doan))

# Goi y 10 phim tot nhat cho moi nguoi dung
goi_y_user = als_model.recommendForAllUsers(10)
goi_y_user.show(5, truncate=False)

# Goi y 10 nguoi dung tiem nang nhat cho moi phim
goi_y_phim = als_model.recommendForAllItems(10)

## Đoạn mã 7.5. Khung tính bảng RFM và vòng lặp chọn K.


> Khung lời giải Bài 7.1 giả định đã có `don_hang` và `df_features` – ô chuẩn bị bên dưới đọc `data/don_hang.csv` và dựng `df_features` (RFM đã chuẩn hóa) để chạy thử.


In [ ]:
# Chuan bi: don_hang tu data/don_hang.csv; df_features = bang RFM da chuan hoa (assembler + scaler)
from pyspark.ml.feature import VectorAssembler, StandardScaler
don_hang = spark.read.csv("data/don_hang.csv", header=True, inferSchema=True)
_rfm = don_hang.groupBy("ma_kh").agg(
    F.datediff(F.lit("2026-07-01"), F.max("ngay_dat")).alias("recency"),
    F.count("*").alias("frequency"), F.sum("gia_tri").alias("monetary"))
_asm = VectorAssembler(inputCols=["recency", "frequency", "monetary"], outputCol="features_raw")
_scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
df_features = _scaler.fit(_asm.transform(_rfm)).transform(_asm.transform(_rfm)).cache()
print(df_features.count(), "khach hang trong bang RFM")

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

rfm = don_hang.groupBy("ma_kh").agg(
    F.datediff(F.lit("2026-07-01"), F.max("ngay_dat"))
     .alias("recency"),
    F.count("*").alias("frequency"),
    F.sum("gia_tri").alias("monetary"))

# Sau khi co df_features (assembler + scaler), thu lan luot cac K
evaluator = ClusteringEvaluator(metricName="silhouette")
for k in range(2, 9):
    model = KMeans(featuresCol="features", k=k,
                   seed=42).fit(df_features)
    sil = evaluator.evaluate(model.transform(df_features))
    print(k, model.summary.trainingCost, sil)

## Đoạn mã 7.6. Khung lời giải bài tập ALS.


In [ ]:
from pyspark.ml.recommendation import ALS

ratings = spark.createDataFrame([
    (1, 101, 5.0), (1, 102, 4.5), (1, 103, 1.0), (1, 105, 4.0),
    (2, 101, 4.5), (2, 102, 4.0), (2, 104, 1.5),
    (3, 102, 1.0), (3, 103, 4.5), (3, 104, 5.0), (3, 105, 1.5),
    (4, 101, 1.5), (4, 103, 4.0), (4, 104, 4.5),
], ["userId", "movieId", "rating"])

als = ALS(userCol="userId", itemCol="movieId", ratingCol="rating",
          rank=5, regParam=0.1, maxIter=10,
          coldStartStrategy="drop", seed=42)
als_model = als.fit(ratings)

cap_can_du_doan = spark.createDataFrame(
    [(2, 105), (4, 102)], ["userId", "movieId"])
als_model.transform(cap_can_du_doan).show()

als_model.recommendForAllUsers(2).show(truncate=False)